# Reorganization and renaming of data and metadata for the isolates sequenced by TBPortals + TGEN

# import statements

In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [2]:
import os

In [3]:
import glob

#### Pandas Viewing Settings

In [4]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

# Parse existing metadata

In [5]:
Repo_MainDir = "../.."
Repo_DataDir = f"{Repo_MainDir}/Data"

TBP_PB_CCS_MetaDir = f"{Repo_DataDir}/221017_TBPortals_LRandSR_InputDataTSVs"

TBP_2022_Metadata_AllInitSamples_TSV          = f"{TBP_PB_CCS_MetaDir}/230901.TBPortals.2022.PBCCS.MetaData.QCPass.31CI.tsv"

TBP_2022_Metadata_QCPass_TSV                  = f"{TBP_PB_CCS_MetaDir}/230901.TBPortals.2022.PBCCS.MetaData.QCPass.31CI.tsv"

TBP22_QCPass_31CI_LRandSR_InputData_Paths_TSV = f"{TBP_PB_CCS_MetaDir}/230901.TBPortals.2022.MtbWGS.LRandSR.QCPass.31CI.InputWGS.PATHs.tsv"

 
TBP22_45CI_AsmSummary_TSV_PATH = TBP_PB_CCS_MetaDir + "/220529.TBP2022.45CI.AsmSummary.V1.tsv"
TBP22_31CI_QCPass_AsmSummary_TSV_PATH = TBP_PB_CCS_MetaDir + "/220529.TBP2022.QCPass.31CI.AsmSummary.V1.tsv"

TBP_29CI_FinalSet_AsmStatsAndQC_Dir = f"{Repo_DataDir}/231002.InputAsmTSVs.TBP22.29I.Complete" 


In [6]:
!head -n 2 $TBP22_QCPass_31CI_LRandSR_InputData_Paths_TSV

SampleID	PacBio_FQ_PATH	Illumina_PE_FQs_PATH	Dataset_Tag	TGEN_SampleID	SeqReason
TB6733	/n/data1/hms/dbmi/farhat/mm774/DownloadedData/2022_TB_Portals_PB_Data/DATA_P7529_20221003/DNA0428/AYW0037_1/CCS_1780_bc1010_BAK8A_OA/demultiplex.bc1010_BAK8A_OA--bc1010_BAK8A_OA.hifi_reads.fastq.gz	/n/data1/hms/dbmi/farhat/mm774/DownloadedData/221017_TBPortals_Tgen_Selected_SR/DNA0428/SRR10379945_1.fastq.gz;/n/data1/hms/dbmi/farhat/mm774/DownloadedData/221017_TBPortals_Tgen_Selected_SR/DNA0428/SRR10379945_2.fastq.gz	TBPortals_2022_PassQC	DNA0428	PutativeRecomb


In [7]:
TBP_2022_QCPass_SampleInfo_DF = pd.read_csv(TBP_2022_Metadata_QCPass_TSV, sep = "\t")
TBP_2022_QCPass_SampleInfo_DF.shape

(31, 11)

In [8]:
TGEN_To_TBP_SampleID_Dict = dict(TBP_2022_QCPass_SampleInfo_DF[['TGEN_SampleID', 'TBP_SampleID']].values)
TBP_To_TGEN_SampleID_Dict = dict(TBP_2022_QCPass_SampleInfo_DF[['TBP_SampleID', 'TGEN_SampleID']].values)
TGEN_To_SR_SRA_RunAcc_Dict = dict(TBP_2022_QCPass_SampleInfo_DF[['TGEN_SampleID', 'SRA_RunAcc_SR']].values)
TBP_To_SR_SRA_RunAcc_Dict = dict(TBP_2022_QCPass_SampleInfo_DF[['TBP_SampleID', 'SRA_RunAcc_SR']].values)


In [9]:
TBP22_31CI_WGS_ReadPaths_DF = pd.read_csv(TBP22_QCPass_31CI_LRandSR_InputData_Paths_TSV, sep = "\t")
TBP22_31CI_WGS_ReadPaths_DF["SRA_RunAcc_SR"] = TBP22_31CI_WGS_ReadPaths_DF["TGEN_SampleID"].map(TGEN_To_SR_SRA_RunAcc_Dict)       

TBP22_31CI_WGS_ReadPaths_DF.shape

(31, 7)

In [10]:
TBP22_31CI_WGS_ReadPaths_DF.head()

,SampleID,PacBio_FQ_PATH,Illumina_PE_FQs_PATH,Dataset_Tag,TGEN_SampleID,SeqReason,SRA_RunAcc_SR
0,TB6733,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,TBPortals_2022_PassQC,DNA0428,PutativeRecomb,SRR10379945
1,TB3898,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,TBPortals_2022_PassQC,DNA373,PutativeRecomb,SRR10397263
2,TB7340,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,TBPortals_2022_PassQC,DNA146,PutativeRecomb,SRR10380093
3,TB6977,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,TBPortals_2022_PassQC,DNA0530,PutativeRecomb,SRR10380218
4,TB6964,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,TBPortals_2022_PassQC,DNA233,PutativeRecomb,SRR10380227


## parse complete HybridAsm QC summary for the 31CI

In [11]:
TBP22_31CI_QCPass_AsmSummary = pd.read_csv(TBP22_31CI_QCPass_AsmSummary_TSV_PATH, sep = "\t")

TBP22_31CI_QCPass_AsmSummary = TBP22_31CI_QCPass_AsmSummary.sort_values("Lineage_AsmPP")

TBP22_31CI_QCPass_AsmSummary["TBP_SampleID"] = TBP22_31CI_QCPass_AsmSummary["SampleID"].map(TGEN_To_TBP_SampleID_Dict)

print(TBP22_31CI_QCPass_AsmSummary.shape)

TBP22_29CI_QCPass_1Contig_DF = TBP22_31CI_QCPass_AsmSummary.query("numContigs_Complete == 1")

TBP22_29CI_QCPass_1Contig_SampleIDs = list( TBP22_29CI_QCPass_1Contig_DF["TBP_SampleID"].values )


print(TBP22_29CI_QCPass_1Contig_DF.shape)


(31, 29)
(29, 29)


In [12]:
TBP22_31CI_QCPass_AsmSummary.head(3)

,SampleID,numContigs_Complete,circContig_Length,circContig_Cov,Flye_EstimatedCov,Flye_ReadLen_N50,Flye_ReadLen_N90,Lineage_Asm,Lineage_AsmPP,FlyeI3_dnaA_Found,FlyeI3M_dnaA_Found,FlyeI3MPP_dnaA_Found,IlluminaCov_To_ONTAsm,NumChanges_PilonPolished,NumSNPs_PilonPolished,NumTotalInsertions_PilonPolished,Num1bpInsertion_PilonPolished,Num2bpInsertion_PilonPolished,NumTotalDeletions_PilonPolished,Num1bpDeletion_PilonPolished,PrimaryLineage_Asm,Dataset_Tag,SR_SRA_RunAcc,SeqReason,PB_SeqRunName,EventID,Event_Gene(s),Event_Relationship,TBP_SampleID
0,DNA0551,1,4410903,187,190,8609,7014,lineage2.2.1,lineage2.2.1,True,NaN,NaN,NaN,0,0,0,0,0,0,0,lineage2,TBPortals_2022_Set3,SRR10380186,MtbPhyloDiversity,P7544,NAN,NAN,NAN,TB7198
1,DNA041,0,0,0,530,4143,2664,lineage2.2.1,lineage2.2.1,True,NaN,NaN,NaN,5,1,3,2,0,1,1,lineage2,TBPortals_2022_Set3,SRR10430372,MtbPhyloDiversity,P7761,NAN,NAN,NAN,TB4570
2,DNA621,1,4412093,127,129,4204,2724,lineage2.2.1,lineage2.2.1,True,NaN,NaN,NaN,1,0,0,0,0,1,0,lineage2,TBPortals_2022_Set3,SRR10397096,PutativeRecomb,P7529,Event_006,"Rv0979c,rpmF,PE_PGRS18",Outside_Event,TB3706


In [13]:
TBP22_31CI_QCPass_AsmSummary.query("numContigs_Complete != 1")[["SampleID", "TBP_SampleID", "SeqReason"]]

,SampleID,TBP_SampleID,SeqReason
1,DNA041,TB4570,MtbPhyloDiversity
22,DNA233,TB6964,PutativeRecomb


#### Save TSV of 29CI that pass ALL levels of QC (This will be our final set)

In [14]:
TBP_29CI_FinalSet_AsmStatsAndQC_Dir = f"{Repo_DataDir}/231002.InputAsmTSVs.TBP22.29I.Complete" 

TBP22_29CI_HybridAsmQCStats_TSV_PATH = TBP_29CI_FinalSet_AsmStatsAndQC_Dir + "/231002.TBP2022.29CI.Final.HybridAsmQCStats.V1.tsv"

TBP22_29CI_QCPass_1Contig_DF.to_csv(TBP22_29CI_HybridAsmQCStats_TSV_PATH, sep = "\t", index=False)


In [15]:
!wc -l $TBP22_29CI_HybridAsmQCStats_TSV_PATH

30 ../../Data/231002.InputAsmTSVs.TBP22.29I.Complete/231002.TBP2022.29CI.Final.HybridAsmQCStats.V1.tsv


In [16]:
!head -n 2 $TBP22_29CI_HybridAsmQCStats_TSV_PATH

SampleID	numContigs_Complete	circContig_Length	circContig_Cov	Flye_EstimatedCov	Flye_ReadLen_N50	Flye_ReadLen_N90	Lineage_Asm	Lineage_AsmPP	FlyeI3_dnaA_Found	FlyeI3M_dnaA_Found	FlyeI3MPP_dnaA_Found	IlluminaCov_To_ONTAsm	NumChanges_PilonPolished	NumSNPs_PilonPolished	NumTotalInsertions_PilonPolished	Num1bpInsertion_PilonPolished	Num2bpInsertion_PilonPolished	NumTotalDeletions_PilonPolished	Num1bpDeletion_PilonPolished	PrimaryLineage_Asm	Dataset_Tag	SR_SRA_RunAcc	SeqReason	PB_SeqRunName	EventID	Event_Gene(s)	Event_Relationship	TBP_SampleID
DNA0551	1	4410903	187	190	8609	7014	lineage2.2.1	lineage2.2.1	True				0	0	0	0	0	0	0	lineage2	TBPortals_2022_Set3	SRR10380186	MtbPhyloDiversity	P7544	NAN	NAN	NAN	TB7198


In [17]:
TBP22_29CI_QCPass_1Contig_DF

,SampleID,numContigs_Complete,circContig_Length,circContig_Cov,Flye_EstimatedCov,Flye_ReadLen_N50,Flye_ReadLen_N90,Lineage_Asm,Lineage_AsmPP,FlyeI3_dnaA_Found,FlyeI3M_dnaA_Found,FlyeI3MPP_dnaA_Found,IlluminaCov_To_ONTAsm,NumChanges_PilonPolished,NumSNPs_PilonPolished,NumTotalInsertions_PilonPolished,Num1bpInsertion_PilonPolished,Num2bpInsertion_PilonPolished,NumTotalDeletions_PilonPolished,Num1bpDeletion_PilonPolished,PrimaryLineage_Asm,Dataset_Tag,SR_SRA_RunAcc,SeqReason,PB_SeqRunName,EventID,Event_Gene(s),Event_Relationship,TBP_SampleID
0,DNA0551,1,4410903,187,190,8609,7014,lineage2.2.1,lineage2.2.1,True,NaN,NaN,NaN,0,0,0,0,0,0,0,lineage2,TBPortals_2022_Set3,SRR10380186,MtbPhyloDiversity,P7544,NAN,NAN,NAN,TB7198
2,DNA621,1,4412093,127,129,4204,2724,lineage2.2.1,lineage2.2.1,True,NaN,NaN,NaN,1,0,0,0,0,1,0,lineage2,TBPortals_2022_Set3,SRR10397096,PutativeRecomb,P7529,Event_006,"Rv0979c,rpmF,PE_PGRS18",Outside_Event,TB3706
3,DNA594,1,4416251,161,174,4212,2712,lineage2.2.1,lineage2.2.1,True,NaN,NaN,NaN,2,0,2,0,0,0,0,lineage2,TBPortals_2022_Set3,SRR10397175,PutativeRecomb,P7529,Event_006,"Rv0979c,rpmF,PE_PGRS18",Within_Event,TB3305
4,DNA435,1,4416249,100,102,9925,7187,lineage2.2.1,lineage2.2.1,True,NaN,NaN,NaN,1,0,1,0,0,0,0,lineage2,TBPortals_2022_Set3,SRR10379929,MtbPhyloDiversity,P7544,NAN,NAN,NAN,TB6760
18,DNA0414,1,4426359,225,226,5817,3402,lineage4.1.2.1,lineage4.1.2.1,True,NaN,NaN,NaN,2,0,2,0,0,0,0,lineage4,TBPortals_2022_Set3,SRR10379962,PutativeRecomb,P7761,Event_017 | Event_023,"ppsB | PPE56,Rv3351c",Outside_Event | Outside_Event,TB6595
23,DNA0432,1,4417503,307,309,7792,6519,lineage4.1.2.1,lineage4.1.2.1,True,NaN,NaN,NaN,1,0,1,0,0,0,0,lineage4,TBPortals_2022_Set3,SRR10379935,PutativeRecomb,P7559,Event_007,Rv1148c,Within_Event,TB6755
14,DNA177,1,4424467,525,527,7948,6551,lineage4.1.2.1,lineage4.1.2.1,True,NaN,NaN,NaN,2,0,2,0,0,0,0,lineage4,TBPortals_2022_Set3,SRR10380054,MtbPhyloDiversity,P7559,NAN,NAN,NAN,TB7379
26,DNA199,1,4408536,151,151,8683,7029,lineage4.1.2.1,lineage4.1.2.1,True,NaN,NaN,NaN,0,0,0,0,0,0,0,lineage4,TBPortals_2022_Set3,SRR10380108,PutativeRecomb,P7544,Event_010,"PPE18,esxK,esxL",Within_Event,TB6552
25,DNA0441,1,4386061,82,82,8459,6983,lineage4.1.2.1,lineage4.1.2.1,True,NaN,NaN,NaN,0,0,0,0,0,0,0,lineage4,TBPortals_2022_Set3,SRR10379924,PutativeRecomb,P7544,Event_010,"PPE18,esxK,esxL",Outside_Event,TB6765
20,DNA146,1,4438452,123,124,8542,7006,lineage4.2.1,lineage4.2.1,True,NaN,NaN,NaN,1,1,0,0,0,0,0,lineage4,TBPortals_2022_Set3,SRR10380093,PutativeRecomb,P7544,Event_002 | Event_024,"Rv0393 | PPE59,Rv3430c",Outside_Event | Within_Event,TB7340


In [18]:
#SRR10397107

# Explore Reseq Reason DF for each sample ID

In [19]:
TBP_2022_QCPass_SampleInfo_DF.shape

(31, 11)

In [20]:
TBP_2022_QCPass_SampleInfo_DF[~TBP_2022_QCPass_SampleInfo_DF["TBP_SampleID"].isin(TBP22_29CI_QCPass_1Contig_SampleIDs)].query("Seq_Group == 'PutativeRecomb' ").shape

(1, 11)

In [21]:
TBP_2022_QCPass_SampleInfo_DF[TBP_2022_QCPass_SampleInfo_DF["TBP_SampleID"].isin(TBP22_29CI_QCPass_1Contig_SampleIDs)].query("Seq_Group == 'PutativeRecomb' ").shape

(23, 11)

In [22]:
Z = TBP_2022_QCPass_SampleInfo_DF[TBP_2022_QCPass_SampleInfo_DF["TBP_SampleID"].isin(TBP22_29CI_QCPass_1Contig_SampleIDs)].query("Seq_Group == 'PutativeRecomb' ")

In [23]:
Z[Z["EventID"].str.contains("Event_001")] # Looks good

,TGEN_SampleID,TBP_SampleID,SRA_RunAcc_SR,Seq_Group,EventID,Event_Gene(s),Event_Relationship,Sequenced?,PB_SeqRunName,Note,Comments/Special Instructions
0,DNA0428,TB6733,SRR10379945,PutativeRecomb,Event_001,"vapC25,vapB25",Within_Event,Y,P7529,NaN,AltId: PBO_SRR10379945
1,DNA373,TB3898,SRR10397263,PutativeRecomb,Event_001,"vapC25,vapB25",Outside_Event,Y,P7529,NaN,AltId: PBO_SRR10397263


In [24]:
Z[Z["EventID"].str.contains("Event_003")] # We lost the isoalte that was sequenced to verify the event

,TGEN_SampleID,TBP_SampleID,SRA_RunAcc_SR,Seq_Group,EventID,Event_Gene(s),Event_Relationship,Sequenced?,PB_SeqRunName,Note,Comments/Special Instructions
3,DNA0530,TB6977,SRR10380218,PutativeRecomb,Event_003,"vapB31,vapC31",Outside_Event,Y,P7544,NaN,AltId: PBO_SRR10380218


In [25]:
Z[Z["EventID"].str.contains("Event_006")] # Looks good

,TGEN_SampleID,TBP_SampleID,SRA_RunAcc_SR,Seq_Group,EventID,Event_Gene(s),Event_Relationship,Sequenced?,PB_SeqRunName,Note,Comments/Special Instructions
5,DNA594,TB3305,SRR10397175,PutativeRecomb,Event_006,"Rv0979c,rpmF,PE_PGRS18",Within_Event,Y,P7529,NaN,AltId: PBO_SRR10397175
6,DNA621,TB3706,SRR10397096,PutativeRecomb,Event_006,"Rv0979c,rpmF,PE_PGRS18",Outside_Event,Y,P7529,NaN,AltId: PBO_SRR10397096


In [26]:
Z[Z["EventID"].str.contains("Event_007")] # Looks good

,TGEN_SampleID,TBP_SampleID,SRA_RunAcc_SR,Seq_Group,EventID,Event_Gene(s),Event_Relationship,Sequenced?,PB_SeqRunName,Note,Comments/Special Instructions
7,DNA0432,TB6755,SRR10379935,PutativeRecomb,Event_007,Rv1148c,Within_Event,Y,P7559,NaN,AltId: PBO_SRR10379935
8,DNA246,TB7044,SRR10380192,PutativeRecomb,Event_007 | Event_008,Rv1148c | PPE18,Outside_Event | Outside_Event,Y,P7544,NaN,AltId: PBO_SRR10380192


In [27]:
Z[Z["EventID"].str.contains("Event_010")] # Looks good

,TGEN_SampleID,TBP_SampleID,SRA_RunAcc_SR,Seq_Group,EventID,Event_Gene(s),Event_Relationship,Sequenced?,PB_SeqRunName,Note,Comments/Special Instructions
9,DNA0441,TB6765,SRR10379924,PutativeRecomb,Event_010,"PPE18,esxK,esxL",Outside_Event,Y,P7544,NaN,AltId: PBO_SRR10379924
10,DNA199,TB6552,SRR10380108,PutativeRecomb,Event_010,"PPE18,esxK,esxL",Within_Event,Y,P7544,NaN,AltId: PBO_SRR10380108


In [28]:
Z[Z["EventID"].str.contains("Event_021")] # Looks good

,TGEN_SampleID,TBP_SampleID,SRA_RunAcc_SR,Seq_Group,EventID,Event_Gene(s),Event_Relationship,Sequenced?,PB_SeqRunName,Note,Comments/Special Instructions
19,DNA361,TB3572,SRR10397163,PutativeRecomb,Event_021,Rv3108,Within_Event,Y,P7544,NaN,AltId: PBO_SRR10397163


In [29]:
Z[Z["EventID"].str.contains("Event_019")] # Looks good

,TGEN_SampleID,TBP_SampleID,SRA_RunAcc_SR,Seq_Group,EventID,Event_Gene(s),Event_Relationship,Sequenced?,PB_SeqRunName,Note,Comments/Special Instructions
18,DNA0527,TB6976,SRR10380219,PutativeRecomb,Event_019,PPE46,Outside_Event,Y,P7559,NaN,AltId: PBO_SRR10380219


In [30]:
Z[Z["EventID"].str.contains("Event_002")]

,TGEN_SampleID,TBP_SampleID,SRA_RunAcc_SR,Seq_Group,EventID,Event_Gene(s),Event_Relationship,Sequenced?,PB_SeqRunName,Note,Comments/Special Instructions
2,DNA146,TB7340,SRR10380093,PutativeRecomb,Event_002 | Event_024,"Rv0393 | PPE59,Rv3430c",Outside_Event | Within_Event,Y,P7544,NaN,AltId: PBO_SRR10380093


In [31]:
Z[Z["EventID"].str.contains("Event_004")]

,TGEN_SampleID,TBP_SampleID,SRA_RunAcc_SR,Seq_Group,EventID,Event_Gene(s),Event_Relationship,Sequenced?,PB_SeqRunName,Note,Comments/Special Instructions


In [32]:
Z[Z["EventID"].str.contains("Event_005")]

,TGEN_SampleID,TBP_SampleID,SRA_RunAcc_SR,Seq_Group,EventID,Event_Gene(s),Event_Relationship,Sequenced?,PB_SeqRunName,Note,Comments/Special Instructions


In [33]:
Z[Z["EventID"].str.contains("Event_006")]

,TGEN_SampleID,TBP_SampleID,SRA_RunAcc_SR,Seq_Group,EventID,Event_Gene(s),Event_Relationship,Sequenced?,PB_SeqRunName,Note,Comments/Special Instructions
5,DNA594,TB3305,SRR10397175,PutativeRecomb,Event_006,"Rv0979c,rpmF,PE_PGRS18",Within_Event,Y,P7529,NaN,AltId: PBO_SRR10397175
6,DNA621,TB3706,SRR10397096,PutativeRecomb,Event_006,"Rv0979c,rpmF,PE_PGRS18",Outside_Event,Y,P7529,NaN,AltId: PBO_SRR10397096


In [34]:
Z[Z["EventID"].str.contains("Event_007")]

,TGEN_SampleID,TBP_SampleID,SRA_RunAcc_SR,Seq_Group,EventID,Event_Gene(s),Event_Relationship,Sequenced?,PB_SeqRunName,Note,Comments/Special Instructions
7,DNA0432,TB6755,SRR10379935,PutativeRecomb,Event_007,Rv1148c,Within_Event,Y,P7559,NaN,AltId: PBO_SRR10379935
8,DNA246,TB7044,SRR10380192,PutativeRecomb,Event_007 | Event_008,Rv1148c | PPE18,Outside_Event | Outside_Event,Y,P7544,NaN,AltId: PBO_SRR10380192


In [35]:
Z[Z["EventID"].str.contains("Event_008")]

,TGEN_SampleID,TBP_SampleID,SRA_RunAcc_SR,Seq_Group,EventID,Event_Gene(s),Event_Relationship,Sequenced?,PB_SeqRunName,Note,Comments/Special Instructions
8,DNA246,TB7044,SRR10380192,PutativeRecomb,Event_007 | Event_008,Rv1148c | PPE18,Outside_Event | Outside_Event,Y,P7544,NaN,AltId: PBO_SRR10380192


In [36]:
Z[Z["EventID"].str.contains("Event_009")]

,TGEN_SampleID,TBP_SampleID,SRA_RunAcc_SR,Seq_Group,EventID,Event_Gene(s),Event_Relationship,Sequenced?,PB_SeqRunName,Note,Comments/Special Instructions


In [37]:
Z[Z["EventID"].str.contains("Event_010")]

,TGEN_SampleID,TBP_SampleID,SRA_RunAcc_SR,Seq_Group,EventID,Event_Gene(s),Event_Relationship,Sequenced?,PB_SeqRunName,Note,Comments/Special Instructions
9,DNA0441,TB6765,SRR10379924,PutativeRecomb,Event_010,"PPE18,esxK,esxL",Outside_Event,Y,P7544,NaN,AltId: PBO_SRR10379924
10,DNA199,TB6552,SRR10380108,PutativeRecomb,Event_010,"PPE18,esxK,esxL",Within_Event,Y,P7544,NaN,AltId: PBO_SRR10380108


In [38]:
Z[Z["EventID"].str.contains("Event_011")]

,TGEN_SampleID,TBP_SampleID,SRA_RunAcc_SR,Seq_Group,EventID,Event_Gene(s),Event_Relationship,Sequenced?,PB_SeqRunName,Note,Comments/Special Instructions
11,DNA0453,TB6778,SRR10380252,PutativeRecomb,Event_011,PPE18,Within_Event,Y,P7559,NaN,AltId: PBO_SRR10380252


In [39]:
Z[Z["EventID"].str.contains("Event_012")]

,TGEN_SampleID,TBP_SampleID,SRA_RunAcc_SR,Seq_Group,EventID,Event_Gene(s),Event_Relationship,Sequenced?,PB_SeqRunName,Note,Comments/Special Instructions
12,DNA237,TB6973,SRR10380223,PutativeRecomb,Event_012,PPE18,Outside_Event,Y,P7559,NaN,AltId: PBO_SRR10380223


In [40]:
Z[Z["EventID"].str.contains("Event_013")]

,TGEN_SampleID,TBP_SampleID,SRA_RunAcc_SR,Seq_Group,EventID,Event_Gene(s),Event_Relationship,Sequenced?,PB_SeqRunName,Note,Comments/Special Instructions
13,DNA216,TB6786,SRR10380244,PutativeRecomb,Event_013,PPE19,Within_Event,Y,P7559,NaN,AltId: PBO_SRR10380244
14,DNA350,TB3256,SRR10397205,PutativeRecomb,Event_013,PPE19,Outside_Event,Y,P7559,NaN,AltId: PBO_SRR10397205


In [41]:
Z[Z["EventID"].str.contains("Event_014")]

,TGEN_SampleID,TBP_SampleID,SRA_RunAcc_SR,Seq_Group,EventID,Event_Gene(s),Event_Relationship,Sequenced?,PB_SeqRunName,Note,Comments/Special Instructions


# Create dictionary of PATHs for ALL 29 final complete Hybrid Asms (From TBP22 dataset)

### Define directories to PMP-SM (PacBio assembly and analysis pipeline)

In [42]:
### Define directories to (PacBio assembly and analysis pipeline)

PacBio_ProjectDir = "/n/data1/hms/dbmi/farhat/mm774/Projects/PacBio_Evaluation_Project"

PMP_SM_Outputs_Dir = PacBio_ProjectDir + "/PacmanPipe_SM_Outputs"

TBP22_HiFi_Asm_OutputDir = PMP_SM_Outputs_Dir + "/230921_TBPortals22_PBCCS_V2"

In [43]:
#!ls -1 $TBP22_HiFi_Asm_OutputDir

### Create dictionary defining paths to each Assembly FASTA (For `TBP22-29CI`)

In [44]:

TBPID_ToAsmFA_OG_Dict = {}

# A) TB Portals 29 Isolates that will be used for Ground-Truth Generation

for SampleID in TBP22_29CI_QCPass_1Contig_SampleIDs:

    # Defining PATHs for assembly pipe output
    Sample_Output_Dir = TBP22_HiFi_Asm_OutputDir + "/" + SampleID
    
    # Define path to PB de novo assembly (No SR Polishing)
    FlyeAsm_I3_Filt_Dir = f"{Sample_Output_Dir}/PB/Flye_Assembly_RenamedAndLengthFiltered"
    PB_CCS_FlyeAsm_I3_FA = f"{FlyeAsm_I3_Filt_Dir}/{SampleID}.flyeassembly.I3.Renamed.100Kb.fasta"
    
    # Define path to PB de novo assembly (No SR Polishing)
    PilonPolish_FlyeI3Asm_Dir =  f"{Sample_Output_Dir}/FlyeAssembly_I3_PilonPolishing/pilon_IllPE_Polishing_I3_Asm_ChangeSNPsINDELsOnly"    
    PB_CCS_FlyeAsm_I3PP_FA = f"{PilonPolish_FlyeI3Asm_Dir}/{SampleID}.Flye.I3Asm.PilonPolished.fasta"
    

    TBPID_ToAsmFA_OG_Dict[SampleID] = PB_CCS_FlyeAsm_I3PP_FA # Use the Pilon Polished version of the Assembly   


### Peak at a specific FASTA's contents

In [45]:
i_TestAsmFA_PATH = TBPID_ToAsmFA_OG_Dict["TB7198"]

In [46]:
!ls -alh $i_TestAsmFA_PATH

-rw-rw-r-- 1 mm774 hpc_farhat 4.3M Sep 21  2023 /n/data1/hms/dbmi/farhat/mm774/Projects/PacBio_Evaluation_Project/PacmanPipe_SM_Outputs/230921_TBPortals22_PBCCS_V2/TB7198/FlyeAssembly_I3_PilonPolishing/pilon_IllPE_Polishing_I3_Asm_ChangeSNPsINDELsOnly/TB7198.Flye.I3Asm.PilonPolished.fasta


In [47]:
!head -n 4 $i_TestAsmFA_PATH

>TB7198_contig_1_pilon
TTGACCGATGACCCCGGTTCAGGCTTCACCACAGTGTGGAACGCGGTCGTCTCCGAACTTAACGGCGACCCTAAGGTTGA
CGACGGACCCAGCAGTGATGCTAATCTCAGCGCTCCGCTGACCCCTCAGCAAAGGGCTTGGCTCAATCTCGTCCAGCCAT
TGACCATCGTCGAGGGGTTTGCTCTGTTATCCGTGCCGAGCAGCTTTGTCCAAAACGAAATCGAGCGCCATCTGCGGGCC


### Peak at dictionary of Hybrid Asm FASTA paths (For TBP22-29CI dataset)

In [48]:
TBPID_ToAsmFA_OG_Dict.keys()

dict_keys(['TB7198', 'TB3706', 'TB3305', 'TB6760', 'TB6595', 'TB6755', 'TB7379', 'TB6552', 'TB6765', 'TB7340', 'TB7396', 'TB6778', 'TB7044', 'TB6973', 'TB6807', 'TB6976', 'TB3572', 'TB6599', 'TB6977', 'TB6830', 'TB6846', 'TB4414', 'TB6786', 'TB6816', 'TB6733', 'TB8073', 'TB6596', 'TB3898', 'TB3256'])

# Create DF of all PB+Illumina reads + assembly paths for the 29 final isolates

In [49]:
TBP22_29CI_WGS_And_Asm_Paths_DF = TBP22_31CI_WGS_ReadPaths_DF[ TBP22_31CI_WGS_ReadPaths_DF["SampleID"].isin(TBP22_29CI_QCPass_1Contig_SampleIDs) ]

TBP22_29CI_WGS_And_Asm_Paths_DF["Genome_ASM_PATH"] = TBP22_29CI_WGS_And_Asm_Paths_DF["SampleID"].map(TBPID_ToAsmFA_OG_Dict)
TBP22_29CI_WGS_And_Asm_Paths_DF.shape

/tmp/ipykernel_4060963/2056992044.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  TBP22_29CI_WGS_And_Asm_Paths_DF["Genome_ASM_PATH"] = TBP22_29CI_WGS_And_Asm_Paths_DF["SampleID"].map(TBPID_ToAsmFA_OG_Dict)


(29, 8)

In [50]:
TBP22_29CI_WGS_And_Asm_Paths_DF.head(1)

,SampleID,PacBio_FQ_PATH,Illumina_PE_FQs_PATH,Dataset_Tag,TGEN_SampleID,SeqReason,SRA_RunAcc_SR,Genome_ASM_PATH
0,TB6733,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,TBPortals_2022_PassQC,DNA0428,PutativeRecomb,SRR10379945,/n/data1/hms/dbmi/farhat/mm774/Projects/PacBio...


# Organize WGS data + assembly for all `TBP22-29CI` isolate

In [51]:
O2_MainDataDir = "/n/data1/hms/dbmi/farhat/mm774/DownloadedData"

TBP22_29CI_DataReorg_MainDir = f"{O2_MainDataDir}/250725.TBP2022.FinalSet_29CI.WGSData"

TBP22_HybridAsms_Dir = f"{TBP22_29CI_DataReorg_MainDir}/TBP22.HybridAsms"

TBP22_WGSData_Dir    = f"{TBP22_29CI_DataReorg_MainDir}/TBP22.WGSData"

!mkdir $TBP22_HybridAsms_Dir
!mkdir $TBP22_WGSData_Dir


mkdir: cannot create directory ‘/n/data1/hms/dbmi/farhat/mm774/DownloadedData/250725.TBP2022.FinalSet_29CI.WGSData/TBP22.HybridAsms’: File exists
mkdir: cannot create directory ‘/n/data1/hms/dbmi/farhat/mm774/DownloadedData/250725.TBP2022.FinalSet_29CI.WGSData/TBP22.WGSData’: File exists


In [52]:
!ls -lah $TBP22_29CI_DataReorg_MainDir

total 0
drwxr-sr-x 2 mm774 hpc_farhat 4.0K Jul 25 14:29 .
drwxrwsr-x 2 mm774 hpc_farhat 4.0K Jul 25 14:12 ..
drwxr-sr-x 2 mm774 hpc_farhat 4.0K Jul 25 14:54 TBP22.HybridAsms
drwxr-sr-x 2 mm774 hpc_farhat 4.0K Jul 25 14:49 TBP22.WGSData


In [53]:
#!ls -lah $TBP22_29CI_DataReorg_MainDir/TBP22.HybridAsms

In [54]:
TBP22_29CI_WGS_And_Asm_Paths_DF.head(1)

,SampleID,PacBio_FQ_PATH,Illumina_PE_FQs_PATH,Dataset_Tag,TGEN_SampleID,SeqReason,SRA_RunAcc_SR,Genome_ASM_PATH
0,TB6733,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,TBPortals_2022_PassQC,DNA0428,PutativeRecomb,SRR10379945,/n/data1/hms/dbmi/farhat/mm774/Projects/PacBio...


## Copy all files to new directory

In [122]:
for i, row in tqdm(TBP22_29CI_WGS_And_Asm_Paths_DF.iterrows()):
    
    i_SampleID = row["SampleID"]
    i_Dataset_Tag = row["Dataset_Tag"]
    i_Sample_SRWGS_SRA_RunAcc = TBP_To_SR_SRA_RunAcc_Dict[i_SampleID]

    i_HybridAsm_FA_PATH = row["Genome_ASM_PATH"]

    i_PacBio_CCS_Reads_FASTQ_PATH = row["PacBio_FQ_PATH"]

    i_Illumina_PE_R1_FASTQ_PATH = row["Illumina_PE_FQs_PATH"].split(";")[0]
    i_Illumina_PE_R2_FASTQ_PATH = row["Illumina_PE_FQs_PATH"].split(";")[1]


    print(i, "-", i_SampleID)
    
    # Make Sample's sub-directory for WGS reads
    o_WGS_Reads_Subdir = f"{TBP22_WGSData_Dir}/{i_SampleID}"

    !mkdir $o_WGS_Reads_Subdir
    
    # Define paths for new FASTQs (PacBio + Illumina)
    o_PacBioCCS_WGS_FASTQ   = f"{o_WGS_Reads_Subdir}/{i_SampleID}.PacBioCCS.fastq.gz"
    o_Illumina_WGS_R1_FASTQ = f"{o_WGS_Reads_Subdir}/{i_SampleID}.{i_Sample_SRWGS_SRA_RunAcc}.IlluminaPE.R1.fastq.gz"
    o_Illumina_WGS_R2_FASTQ = f"{o_WGS_Reads_Subdir}/{i_SampleID}.{i_Sample_SRWGS_SRA_RunAcc}.IlluminaPE.R2.fastq.gz"

    # Define path for renamed Hybrid Assembly FASTA
    o_HybridAsm_FA_PATH = f"{TBP22_HybridAsms_Dir}/{i_SampleID}.HybridAsm.fasta"

    # Copy Illumina read FASTQs
    !cp $i_Illumina_PE_R1_FASTQ_PATH $o_Illumina_WGS_R1_FASTQ
    !cp $i_Illumina_PE_R2_FASTQ_PATH $o_Illumina_WGS_R2_FASTQ

    # Copy PacBio CCS (HiFi) read FASTQs
    !cp $i_PacBio_CCS_Reads_FASTQ_PATH $o_PacBioCCS_WGS_FASTQ

    # Copy complete Hybrid Genome assembly
    !cp $i_HybridAsm_FA_PATH $o_HybridAsm_FA_PATH
    
    # print(o_HybridAsm_FA_PATH)
    # print(o_PacBioCCS_WGS_FASTQ)
    # print(o_Illumina_WGS_R1_FASTQ)
    # print(o_Illumina_WGS_R2_FASTQ)
    # print("-------------\n")
 
    #break 


0it [00:00, ?it/s]

0 - TB6733
mkdir: cannot create directory ‘/n/data1/hms/dbmi/farhat/mm774/DownloadedData/250725.TBP2022.FinalSet_29CI.WGSData/TBP22.WGSData/TB6733’: File exists


1it [00:05,  5.44s/it]

1 - TB3898
mkdir: cannot create directory ‘/n/data1/hms/dbmi/farhat/mm774/DownloadedData/250725.TBP2022.FinalSet_29CI.WGSData/TBP22.WGSData/TB3898’: File exists


2it [00:07,  3.73s/it]

...

27it [01:55,  4.68s/it]

29 - TB6760
mkdir: cannot create directory ‘/n/data1/hms/dbmi/farhat/mm774/DownloadedData/250725.TBP2022.FinalSet_29CI.WGSData/TBP22.WGSData/TB6760’: File exists


28it [01:58,  4.14s/it]

30 - TB7379
mkdir: cannot create directory ‘/n/data1/hms/dbmi/farhat/mm774/DownloadedData/250725.TBP2022.FinalSet_29CI.WGSData/TBP22.WGSData/TB7379’: File exists


29it [02:07,  4.39s/it]


## Create a NEW DF with these new input file paths

In [55]:
TBP22_29CI_UpdatedPathsRowsList = []

for i, row in tqdm(TBP22_29CI_WGS_And_Asm_Paths_DF.iterrows()):
    
    i_SampleID = row["SampleID"]
    i_Dataset_Tag = row["Dataset_Tag"]
    i_Sample_SRWGS_SRA_RunAcc = TBP_To_SR_SRA_RunAcc_Dict[i_SampleID]

    
    # Make Sample's sub-directory for WGS reads
    o_WGS_Reads_Subdir = f"{TBP22_WGSData_Dir}/{i_SampleID}"
    
    # Define paths for new FASTQs (PacBio + Illumina)
    o_PacBioCCS_WGS_FASTQ   = f"{o_WGS_Reads_Subdir}/{i_SampleID}.PacBioCCS.fastq.gz"
    o_Illumina_WGS_R1_FASTQ = f"{o_WGS_Reads_Subdir}/{i_SampleID}.{i_Sample_SRWGS_SRA_RunAcc}.IlluminaPE.R1.fastq.gz"
    o_Illumina_WGS_R2_FASTQ = f"{o_WGS_Reads_Subdir}/{i_SampleID}.{i_Sample_SRWGS_SRA_RunAcc}.IlluminaPE.R2.fastq.gz"

    # Define path for renamed Hybrid Assembly FASTA
    o_HybridAsm_FA_PATH = f"{TBP22_HybridAsms_Dir}/{i_SampleID}.HybridAsm.fasta"


    row["New_Illumina_PE_FQs_PATH"] =  o_Illumina_WGS_R1_FASTQ + ";" + o_Illumina_WGS_R2_FASTQ
    row["New_PacBio_FQ_PATH"]       =  o_PacBioCCS_WGS_FASTQ 
    row["New_HybridAsm_FA_PATH"]    =  o_HybridAsm_FA_PATH

    TBP22_29CI_UpdatedPathsRowsList.append(row)

TBP22_29CI_UpdatedDataPaths_BothOldAndNew_V1_DF = pd.DataFrame(TBP22_29CI_UpdatedPathsRowsList)
TBP22_29CI_UpdatedDataPaths_BothOldAndNew_V1_DF.shape

29it [00:00, 1258.48it/s]


(29, 11)

## `TBP22_29CI_UpdatedDataPaths_V1_DF`

In [56]:
TBP22_29CI_UpdatedDataPaths_V1_DF = TBP22_29CI_UpdatedDataPaths_BothOldAndNew_V1_DF[["SampleID", "TGEN_SampleID",
                                                                                     "SRA_RunAcc_SR",
                                                                                     "Dataset_Tag", "SeqReason",
                                                                                     "New_Illumina_PE_FQs_PATH",
                                                                                     "New_PacBio_FQ_PATH",
                                                                                     "New_HybridAsm_FA_PATH"]]

TBP22_29CI_UpdatedDataPaths_V1_DF.columns = ["SampleID", "TGEN_SampleID",
                                             "SRA_RunAcc_SR",
                                             "Dataset_Tag", "SeqReason",
                                             "Illumina_PE_FQs_PATH",
                                             "PacBio_FQ_PATH",
                                             "HybridAsm_FA_PATH"]

TBP22_29CI_UpdatedDataPaths_V1_DF.shape

(29, 8)

In [57]:
TBP22_29CI_UpdatedDataPaths_V1_DF["SeqReason"].value_counts()

SeqReason
PutativeRecomb       23
MtbPhyloDiversity     6
Name: count, dtype: int64

In [58]:
TBP22_29CI_UpdatedDataPaths_V1_DF.query("SampleID == 'TB6964'")

,SampleID,TGEN_SampleID,SRA_RunAcc_SR,Dataset_Tag,SeqReason,Illumina_PE_FQs_PATH,PacBio_FQ_PATH,HybridAsm_FA_PATH


In [59]:
TBP22_29CI_UpdatedDataPaths_V1_DF.query("SampleID == 'TB6964'")

,SampleID,TGEN_SampleID,SRA_RunAcc_SR,Dataset_Tag,SeqReason,Illumina_PE_FQs_PATH,PacBio_FQ_PATH,HybridAsm_FA_PATH


In [60]:
dictOf_TGEN_GCEventVerf_TBP_ID_Mappings_FINAL = {"Event_001" : {"Target" : "TB6733", "Control" : "TB3898"},
                                                 "Event_003" : {"Target" : "TB6599", "Control" : "TB6977"},
                                                 "Event_006" : {"Target" : "TB3305", "Control" : "TB3706"},
                                                 "Event_007" : {"Target" : "TB6755", "Control" : "TB7044"},
                                                 "Event_010" : {"Target" : "TB6552", "Control" : "TB6765"},
                                                 "Event_011" : {"Target" : "TB6778", "Control" : "TB6973"}, #Note: This is a special case, the CTRL isolate of Event_012 is being used as a CTRL
                                                 "Event_013" : {"Target" : "TB6786", "Control" : "TB3256"},
                                                 "Event_019" : {"Target" : "TB6977", "Control" : "TB6976"}, # This is a special case, the CTRL isolate that was selected for Event_003, is now the EVENT VERF isolate for Event_019
                                                 "Event_021" : {"Target" : "TB3572", "Control" : "TB6976"}, # Note: This is a special case, the CTRL isolate of Event_019 is being used as a CTRL
                                                 "Event_022" : {"Target" : "TB6596", "Control" : "TB8073"},    
                                                 "Event_024" : {"Target" : "TB7340", "Control" : "TB6807"},   
                                                 "Event_025" : {"Target" : "TB6846", "Control" : "TB4414"},  # Note: This is a special case, the CTRL isolate of Event_016 is being used as a CTRL
}

ListOfIsolateIDs_ForVerf = []

Ctr = 1 
for i_EventID, event_IsolateMap in dictOf_TGEN_GCEventVerf_TBP_ID_Mappings_FINAL.items():

    print(Ctr, " - ID", i_EventID)
    i_TarID =  event_IsolateMap["Target"]
    i_CtrlID =  event_IsolateMap["Control"]


    ListOfIsolateIDs_ForVerf.append(i_TarID)
    ListOfIsolateIDs_ForVerf.append(i_CtrlID)

    print("   Verf Isolate:", i_TarID,  TBP22_29CI_UpdatedDataPaths_V1_DF.query(f"SampleID == '{i_TarID}'").shape[0]  ) 
    print("   CTRL Isolate:", i_CtrlID, TBP22_29CI_UpdatedDataPaths_V1_DF.query(f"SampleID == '{i_CtrlID}'").shape[0]  ) 

    print("--------\n")

    Ctr += 1

print(len(ListOfIsolateIDs_ForVerf))
ListOfIsolateIDs_ForVerf = list(set(ListOfIsolateIDs_ForVerf))
print(len(ListOfIsolateIDs_ForVerf))


1  - ID Event_001
   Verf Isolate: TB6733 1
   CTRL Isolate: TB3898 1
--------

2  - ID Event_003
   Verf Isolate: TB6599 1
   CTRL Isolate: TB6977 1
--------

3  - ID Event_006
   Verf Isolate: TB3305 1
   CTRL Isolate: TB3706 1
--------

4  - ID Event_007
   Verf Isolate: TB6755 1
   CTRL Isolate: TB7044 1
--------

5  - ID Event_010
   Verf Isolate: TB6552 1
   CTRL Isolate: TB6765 1
--------

6  - ID Event_011
   Verf Isolate: TB6778 1
   CTRL Isolate: TB6973 1
--------

7  - ID Event_013
   Verf Isolate: TB6786 1
   CTRL Isolate: TB3256 1
--------

8  - ID Event_019
   Verf Isolate: TB6977 1
   CTRL Isolate: TB6976 1
--------

9  - ID Event_021
   Verf Isolate: TB3572 1
   CTRL Isolate: TB6976 1
--------

10  - ID Event_022
   Verf Isolate: TB6596 1
   CTRL Isolate: TB8073 1
--------

11  - ID Event_024
   Verf Isolate: TB7340 1
   CTRL Isolate: TB6807 1
--------

12  - ID Event_025
   Verf Isolate: TB6846 1
   CTRL Isolate: TB4414 1
--------

24
22


In [61]:
ListOfIsolateIDs_ForVerf

['TB6765',
 'TB6552',
 'TB7044',
 'TB6786',
 'TB6733',
 'TB6977',
 'TB6755',
 'TB3256',
 'TB3572',
 'TB7340',
 'TB3706',
 'TB6778',
 'TB6846',
 'TB3898',
 'TB6807',
 'TB8073',
 'TB6976',
 'TB6596',
 'TB6599',
 'TB6973',
 'TB3305',
 'TB4414']

In [62]:
TBP22_29CI_UpdatedDataPaths_V1_DF.head()

,SampleID,TGEN_SampleID,SRA_RunAcc_SR,Dataset_Tag,SeqReason,Illumina_PE_FQs_PATH,PacBio_FQ_PATH,HybridAsm_FA_PATH
0,TB6733,DNA0428,SRR10379945,TBPortals_2022_PassQC,PutativeRecomb,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...
1,TB3898,DNA373,SRR10397263,TBPortals_2022_PassQC,PutativeRecomb,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...
2,TB7340,DNA146,SRR10380093,TBPortals_2022_PassQC,PutativeRecomb,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...
3,TB6977,DNA0530,SRR10380218,TBPortals_2022_PassQC,PutativeRecomb,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...
5,TB3305,DNA594,SRR10397175,TBPortals_2022_PassQC,PutativeRecomb,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...


In [63]:
TBP22_29CI_UpdatedDataPaths_V1_DF[ ~TBP22_29CI_UpdatedDataPaths_V1_DF["SampleID"].isin(ListOfIsolateIDs_ForVerf)]

,SampleID,TGEN_SampleID,SRA_RunAcc_SR,Dataset_Tag,SeqReason,Illumina_PE_FQs_PATH,PacBio_FQ_PATH,HybridAsm_FA_PATH
16,TB6595,DNA0414,SRR10379962,TBPortals_2022_PassQC,PutativeRecomb,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...
25,TB6816,DNA0486,SRR10379988,TBPortals_2022_PassQC,MtbPhyloDiversity,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...
26,TB7198,DNA0551,SRR10380186,TBPortals_2022_PassQC,MtbPhyloDiversity,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...
27,TB6830,DNA224,SRR10379904,TBPortals_2022_PassQC,MtbPhyloDiversity,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...
28,TB7396,DNA287,SRR10380035,TBPortals_2022_PassQC,MtbPhyloDiversity,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...
29,TB6760,DNA435,SRR10379929,TBPortals_2022_PassQC,MtbPhyloDiversity,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...
30,TB7379,DNA177,SRR10380054,TBPortals_2022_PassQC,MtbPhyloDiversity,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...


### Create Input File Paths + Sample Info for final 22CI to be used for event verfication

In [64]:
TBP22_29CI_UpdatedDataPaths_V1_DF[TBP22_29CI_UpdatedDataPaths_V1_DF["SampleID"].isin(ListOfIsolateIDs_ForVerf)].shape

(22, 8)

In [66]:
TBP22_22CI_EventVerfIsolates_UpdatedDataPaths_V1_DF = TBP22_29CI_UpdatedDataPaths_V1_DF[TBP22_29CI_UpdatedDataPaths_V1_DF["SampleID"].isin(ListOfIsolateIDs_ForVerf)]
TBP22_22CI_EventVerfIsolates_UpdatedDataPaths_V1_DF.shape

(22, 8)

### Output to TSV `"Sample Info + Input File Paths"` DF

In [67]:
!pwd

/n/data1/hms/dbmi/farhat/mm774/Projects/Mtb-GeneConv/mtb-GE-analysis/Analysis/7_TGEN_GCVerf_Part2_OrgDataForSelectedIsolates


In [68]:
Repo_MainDir = "../.."
Repo_DataDir = f"{Repo_MainDir}/Data"

TBP_PB_CCS_MetaDir = f"{Repo_DataDir}/221017_TBPortals_LRandSR_InputDataTSVs"

TBP22_Final_QCPass_29CI_Input_AsmAndReads_Paths_TSV = f"{TBP_PB_CCS_MetaDir}/230901.TBP22.QCPass_29CI.All.Asm_LR_SR.InputPATHs.tsv"

TBP22_GCEVerf_Metadata_Dir = f"{Repo_DataDir}/TBP22.22CI.GCEVerfIsolates.Metadata" 


TBP22_Final_QCPass_22CI_Input_AsmAndReads_Paths_TSV = f"{TBP_PB_CCS_MetaDir}/250801.TBP22.22CI.GCEVerfIsolates.Asm_LR_SR.InputPATHs.tsv"

TBP22_Final_QCPass_22CI_Input_AsmAndReads_Paths_AltDir_TSV = f"{TBP22_GCEVerf_Metadata_Dir}/250801.TBP22.22CI.GCEVerfIsolates.Asm_LR_SR.InputPATHs.tsv"



In [69]:
!mkdir $TBP22_GCEVerf_Metadata_Dir

mkdir: cannot create directory ‘../../Data/TBP22.22CI.GCEVerfIsolates.Metadata’: File exists


In [70]:
TBP22_29CI_UpdatedDataPaths_V1_DF.to_csv(TBP22_Final_QCPass_29CI_Input_AsmAndReads_Paths_TSV,
                                         sep ="\t", index=False)


In [71]:
TBP22_22CI_EventVerfIsolates_UpdatedDataPaths_V1_DF.to_csv(TBP22_Final_QCPass_22CI_Input_AsmAndReads_Paths_TSV,
                                         sep ="\t", index=False)


In [72]:
TBP22_22CI_EventVerfIsolates_UpdatedDataPaths_V1_DF.to_csv(TBP22_Final_QCPass_22CI_Input_AsmAndReads_Paths_AltDir_TSV,
                                         sep ="\t", index=False)


In [109]:
#!ls -1 $TBP_PB_CCS_MetaDir

## Create PacBio Hybrid Asm QC Stats DF (FINAL 22CI For EventVerf)

In [79]:
TBP22_22CI_QCPass_DF =    TBP22_29CI_QCPass_1Contig_DF[TBP22_29CI_QCPass_1Contig_DF["TBP_SampleID"].isin(ListOfIsolateIDs_ForVerf) ]
TBP22_22CI_QCPass_DF["TGEN_SampleID"] = TBP22_22CI_QCPass_DF["SampleID"]
TBP22_22CI_QCPass_DF["SampleID"] = TBP22_22CI_QCPass_DF["TBP_SampleID"]
TBP22_22CI_QCPass_DF["Dataset_Tag"] = "TBPortals2022"
TBP22_22CI_QCPass_DF["SeqReason"]   = "ReseqToVerfGCE"


columns_to_drop = [
    'FlyeI3_dnaA_Found', 'FlyeI3M_dnaA_Found', 'FlyeI3MPP_dnaA_Found',
    'IlluminaCov_To_ONTAsm', 'NumChanges_PilonPolished', 'NumSNPs_PilonPolished',
    'NumTotalInsertions_PilonPolished', 'Num1bpInsertion_PilonPolished',
    'Num2bpInsertion_PilonPolished', 'NumTotalDeletions_PilonPolished',
    'Num1bpDeletion_PilonPolished'
]

TBP22_22CI_QCPass_DF = TBP22_22CI_QCPass_DF.drop(columns=columns_to_drop, errors='ignore')


TBP22_22CI_QCPass_DF.shape

/tmp/ipykernel_4060963/3074463763.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  TBP22_22CI_QCPass_DF["TGEN_SampleID"] = TBP22_22CI_QCPass_DF["SampleID"]
/tmp/ipykernel_4060963/3074463763.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  TBP22_22CI_QCPass_DF["SampleID"] = TBP22_22CI_QCPass_DF["TBP_SampleID"]
/tmp/ipykernel_4060963/3074463763.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See th

(22, 19)

In [80]:
TBP22_22CI_QCPass_DF.columns

Index(['SampleID', 'numContigs_Complete', 'circContig_Length', 'circContig_Cov', 'Flye_EstimatedCov', 'Flye_ReadLen_N50', 'Flye_ReadLen_N90', 'Lineage_Asm', 'Lineage_AsmPP', 'PrimaryLineage_Asm', 'Dataset_Tag', 'SR_SRA_RunAcc', 'SeqReason', 'PB_SeqRunName', 'EventID', 'Event_Gene(s)', 'Event_Relationship', 'TBP_SampleID', 'TGEN_SampleID'], dtype='object')

In [81]:
TBP22_22CI_QCPass_DF.head()

,SampleID,numContigs_Complete,circContig_Length,circContig_Cov,Flye_EstimatedCov,Flye_ReadLen_N50,Flye_ReadLen_N90,Lineage_Asm,Lineage_AsmPP,PrimaryLineage_Asm,Dataset_Tag,SR_SRA_RunAcc,SeqReason,PB_SeqRunName,EventID,Event_Gene(s),Event_Relationship,TBP_SampleID,TGEN_SampleID
2,TB3706,1,4412093,127,129,4204,2724,lineage2.2.1,lineage2.2.1,lineage2,TBPortals2022,SRR10397096,ReseqToVerfGCE,P7529,Event_006,"Rv0979c,rpmF,PE_PGRS18",Outside_Event,TB3706,DNA621
3,TB3305,1,4416251,161,174,4212,2712,lineage2.2.1,lineage2.2.1,lineage2,TBPortals2022,SRR10397175,ReseqToVerfGCE,P7529,Event_006,"Rv0979c,rpmF,PE_PGRS18",Within_Event,TB3305,DNA594
23,TB6755,1,4417503,307,309,7792,6519,lineage4.1.2.1,lineage4.1.2.1,lineage4,TBPortals2022,SRR10379935,ReseqToVerfGCE,P7559,Event_007,Rv1148c,Within_Event,TB6755,DNA0432
26,TB6552,1,4408536,151,151,8683,7029,lineage4.1.2.1,lineage4.1.2.1,lineage4,TBPortals2022,SRR10380108,ReseqToVerfGCE,P7544,Event_010,"PPE18,esxK,esxL",Within_Event,TB6552,DNA199
25,TB6765,1,4386061,82,82,8459,6983,lineage4.1.2.1,lineage4.1.2.1,lineage4,TBPortals2022,SRR10379924,ReseqToVerfGCE,P7544,Event_010,"PPE18,esxK,esxL",Outside_Event,TB6765,DNA0441


In [82]:
TBP22_22CI_QCPass_DF

,SampleID,numContigs_Complete,circContig_Length,circContig_Cov,Flye_EstimatedCov,Flye_ReadLen_N50,Flye_ReadLen_N90,Lineage_Asm,Lineage_AsmPP,PrimaryLineage_Asm,Dataset_Tag,SR_SRA_RunAcc,SeqReason,PB_SeqRunName,EventID,Event_Gene(s),Event_Relationship,TBP_SampleID,TGEN_SampleID
2,TB3706,1,4412093,127,129,4204,2724,lineage2.2.1,lineage2.2.1,lineage2,TBPortals2022,SRR10397096,ReseqToVerfGCE,P7529,Event_006,"Rv0979c,rpmF,PE_PGRS18",Outside_Event,TB3706,DNA621
3,TB3305,1,4416251,161,174,4212,2712,lineage2.2.1,lineage2.2.1,lineage2,TBPortals2022,SRR10397175,ReseqToVerfGCE,P7529,Event_006,"Rv0979c,rpmF,PE_PGRS18",Within_Event,TB3305,DNA594
23,TB6755,1,4417503,307,309,7792,6519,lineage4.1.2.1,lineage4.1.2.1,lineage4,TBPortals2022,SRR10379935,ReseqToVerfGCE,P7559,Event_007,Rv1148c,Within_Event,TB6755,DNA0432
26,TB6552,1,4408536,151,151,8683,7029,lineage4.1.2.1,lineage4.1.2.1,lineage4,TBPortals2022,SRR10380108,ReseqToVerfGCE,P7544,Event_010,"PPE18,esxK,esxL",Within_Event,TB6552,DNA199
25,TB6765,1,4386061,82,82,8459,6983,lineage4.1.2.1,lineage4.1.2.1,lineage4,TBPortals2022,SRR10379924,ReseqToVerfGCE,P7544,Event_010,"PPE18,esxK,esxL",Outside_Event,TB6765,DNA0441
20,TB7340,1,4438452,123,124,8542,7006,lineage4.2.1,lineage4.2.1,lineage4,TBPortals2022,SRR10380093,ReseqToVerfGCE,P7544,Event_002 | Event_024,"Rv0393 | PPE59,Rv3430c",Outside_Event | Within_Event,TB7340,DNA146
27,TB6778,1,4426239,147,148,7909,6539,lineage4.2.1,lineage4.2.1,lineage4,TBPortals2022,SRR10380252,ReseqToVerfGCE,P7559,Event_011,PPE18,Within_Event,TB6778,DNA0453
24,TB7044,1,4435184,101,101,8506,6988,lineage4.2.1,lineage4.2.1,lineage4,TBPortals2022,SRR10380192,ReseqToVerfGCE,P7544,Event_007 | Event_008,Rv1148c | PPE18,Outside_Event | Outside_Event,TB7044,DNA246
28,TB6973,1,4431752,101,102,7910,6549,lineage4.2.1,lineage4.2.1,lineage4,TBPortals2022,SRR10380223,ReseqToVerfGCE,P7559,Event_012,PPE18,Outside_Event,TB6973,DNA237
8,TB6807,1,4438385,166,167,8028,6566,lineage4.2.1,lineage4.2.1,lineage4,TBPortals2022,SRR10379998,ReseqToVerfGCE,P7559,Event_024,"PPE59,Rv3430c",Outside_Event,TB6807,DNA0479


## Output PacBio Hybrid Asm QC Stats TSV for the FINAL 22CI (For EventVerf)

In [83]:
!mkdir $TBP22_GCEVerf_Metadata_Dir

mkdir: cannot create directory ‘../../Data/TBP22.22CI.GCEVerfIsolates.Metadata’: File exists


In [84]:
TBP22_GCEVerf_Metadata_Dir = f"{Repo_DataDir}/TBP22.22CI.GCEVerfIsolates.Metadata" 

TBP22_22CI_HybridAsmQCStats_TSV_PATH = TBP22_GCEVerf_Metadata_Dir + "/250801.TBP22.22CI.GCEVerfIsolates.HybridAsmQCStats.V1.tsv"

TBP22_22CI_QCPass_DF.to_csv(TBP22_22CI_HybridAsmQCStats_TSV_PATH, sep = "\t", index=False)


In [85]:
!ls -lah $TBP22_GCEVerf_Metadata_Dir

total 20K
drwxr-sr-x 2 mm774 hpc_farhat 4.0K Aug  1 15:05 .
drwxr-sr-x 2 mm774 hpc_farhat 4.0K Aug  1 14:23 ..
-rw-r--r-- 1 mm774 hpc_farhat  14K Aug  3 14:44 250801.TBP22.22CI.GCEVerfIsolates.Asm_LR_SR.InputPATHs.tsv
-rw-r--r-- 1 mm774 hpc_farhat 3.9K Aug  3 14:44 250801.TBP22.22CI.GCEVerfIsolates.HybridAsmQCStats.V1.tsv
-rw-r--r-- 1 mm774 hpc_farhat  337 Aug  1 15:04 250801.TBP22.22CI.TGENSR_Reseq.GCEvent_To_IsolateIDs.tsv
-rw-r--r-- 1 mm774 hpc_farhat 1.3K Aug  1 15:04 250801.TBP22.22CI.TGENSR_Reseq.IsolateID_To_GCEvents.tsv


In [86]:
!head $TBP22_22CI_HybridAsmQCStats_TSV_PATH

SampleID	numContigs_Complete	circContig_Length	circContig_Cov	Flye_EstimatedCov	Flye_ReadLen_N50	Flye_ReadLen_N90	Lineage_Asm	Lineage_AsmPP	PrimaryLineage_Asm	Dataset_Tag	SR_SRA_RunAcc	SeqReason	PB_SeqRunName	EventID	Event_Gene(s)	Event_Relationship	TBP_SampleID	TGEN_SampleID
TB3706	1	4412093	127	129	4204	2724	lineage2.2.1	lineage2.2.1	lineage2	TBPortals2022	SRR10397096	ReseqToVerfGCE	P7529	Event_006	Rv0979c,rpmF,PE_PGRS18	Outside_Event	TB3706	DNA621
TB3305	1	4416251	161	174	4212	2712	lineage2.2.1	lineage2.2.1	lineage2	TBPortals2022	SRR10397175	ReseqToVerfGCE	P7529	Event_006	Rv0979c,rpmF,PE_PGRS18	Within_Event	TB3305	DNA594
TB6755	1	4417503	307	309	7792	6519	lineage4.1.2.1	lineage4.1.2.1	lineage4	TBPortals2022	SRR10379935	ReseqToVerfGCE	P7559	Event_007	Rv1148c	Within_Event	TB6755	DNA0432
TB6552	1	4408536	151	151	8683	7029	lineage4.1.2.1	lineage4.1.2.1	lineage4	TBPortals2022	SRR10380108	ReseqToVerfGCE	P7544	Event_010	PPE18,esxK,esxL	Within_Event	TB6552	DNA199
TB6765	1	4386061	82	82	8459

# Extras

### Inspect available FASTQ file paths in DFs

In [87]:
TBP22_22CI_EventVerfIsolates_UpdatedDataPaths_V1_DF["PacBio_FQ_PATH"].values[0]

'/n/data1/hms/dbmi/farhat/mm774/DownloadedData/250725.TBP2022.FinalSet_29CI.WGSData/TBP22.WGSData/TB6733/TB6733.PacBioCCS.fastq.gz'

In [ ]:
TBP22_22CI_EventVerfIsolates_UpdatedDataPaths_V1_DF["Illumina_PE_FQs_PATH"].values[0]

In [ ]:
!ls -alh /n/data1/hms/dbmi/farhat/mm774/DownloadedData/2022_TB_Portals_PB_Data/DATA_P7529_20221003/DNA0428/AYW0037_1/CCS_1780_bc1010_BAK8A_OA/demultiplex.bc1010_BAK8A_OA--bc1010_BAK8A_OA.hifi_reads.fastq.gz 

In [ ]:
!ls -alh /n/data1/hms/dbmi/farhat/mm774/DownloadedData/221017_TBPortals_Tgen_Selected_SR/DNA0428/SRR10379945_1.fastq.gz  


In [ ]:
!ls -alh /n/data1/hms/dbmi/farhat/mm774/DownloadedData/221017_TBPortals_Tgen_Selected_SR/DNA0428/SRR10379945_2.fastq.gz  